# IAM Full-Page Handwriting Training (Kaggle)

This notebook trains a separate **line-level model** for full-page OCR.
Your single-word model remains unchanged.

## 1) Kaggle Input Setup

Before running cells:
1. Enable GPU in Notebook settings.
2. Add Kaggle Input dataset: **IAM Handwritten Forms Dataset**.
3. No extra Python script upload is needed (single-file notebook flow).

In [ ]:
# Check Kaggle environment and input datasets
import os

print('Working directory:', os.getcwd())
print('\n/kaggle/input contents:')
for item in sorted(os.listdir('/kaggle/input')):
    print('-', item)

In [ ]:
# Install/upgrade required package (safe on Kaggle)
!pip -q install opencv-python-headless==4.10.0.84

## 2) Define Training Pipeline (Single-File)
This cell embeds the full line-level training pipeline directly in the notebook.

In [ ]:
# Single-file IAM training pipeline (auto-detects lines/words/forms layouts)
import os
import json
import random
from dataclasses import dataclass

import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

PAD_VALUE = -1
DEFAULT_SEED = 42

ALPHABET = (
    " !\"#&'()*+,-./0123456789:;?"
    "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    "abcdefghijklmnopqrstuvwxyz"
)
BLANK_INDEX = len(ALPHABET)

def set_seed(seed: int = DEFAULT_SEED):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

def configure_gpu():
    gpus = tf.config.list_physical_devices("GPU")
    if gpus:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"[GPU] Found {len(gpus)} GPU(s). Memory growth enabled.")
    else:
        print("[GPU] No GPU found. Training will run on CPU.")

def _scan_input_tree(base="/kaggle/input", max_depth=5):
    found_files = []
    found_dirs = []
    base_depth = base.rstrip('/').count('/')
    for root, dirs, files in os.walk(base):
        depth = root.rstrip('/').count('/') - base_depth
        if depth > max_depth:
            continue
        for d in dirs:
            found_dirs.append(os.path.join(root, d))
        for f in files:
            found_files.append(os.path.join(root, f))
    return found_files, found_dirs

def detect_dataset_layout(base="/kaggle/input"):
    files, dirs = _scan_input_tree(base, max_depth=6)

    # Preferred order for full-page task: lines > forms > words
    lines_txt = [p for p in files if os.path.basename(p).lower() == "lines.txt"]
    words_txt = [p for p in files if os.path.basename(p).lower() == "words.txt"]
    forms_txt = [p for p in files if os.path.basename(p).lower() == "forms.txt"]

    lines_dirs = [d for d in dirs if os.path.basename(d).lower() == "lines"]
    words_dirs = [d for d in dirs if os.path.basename(d).lower() == "words"]
    forms_dirs = [d for d in dirs if os.path.basename(d).lower() == "forms"]

    if lines_txt and lines_dirs:
        return {"mode": "lines", "txt": lines_txt[0], "img_dir": lines_dirs[0], "img_h": 64, "img_w": 1024, "max_label_len": 128}
    if forms_txt and forms_dirs:
        return {"mode": "forms", "txt": forms_txt[0], "img_dir": forms_dirs[0], "img_h": 64, "img_w": 1536, "max_label_len": 384}
    if words_txt and words_dirs:
        return {"mode": "words", "txt": words_txt[0], "img_dir": words_dirs[0], "img_h": 64, "img_w": 256, "max_label_len": 48}

    # Generic CSV fallback (must have image+label columns)
    csv_files = [p for p in files if p.lower().endswith('.csv')]
    for csv_path in csv_files:
        try:
            df = pd.read_csv(csv_path, nrows=5)
            cols = {c.lower() for c in df.columns}
            if (("image_path" in cols or "path" in cols or "image" in cols) and ("label" in cols or "text" in cols or "transcript" in cols)):
                return {"mode": "csv", "csv": csv_path, "img_h": 64, "img_w": 1024, "max_label_len": 128}
        except Exception:
            pass

    print("[Detect] Could not auto-detect supported IAM layout.")
    print("[Detect] Found top txt/csv files:")
    for p in sorted([x for x in files if x.lower().endswith((".txt", ".csv"))])[:60]:
        print(" -", p)
    raise FileNotFoundError("No supported annotation layout found (lines/forms/words/csv).")

def parse_iam_txt_layout(txt_path: str, img_dir: str, mode: str):
    rows = []
    skipped_status = 0
    skipped_missing = 0

    with open(txt_path, "r", encoding="utf-8", errors="ignore") as f:
        for raw in f:
            line = raw.strip()
            if not line or line.startswith("#"):
                continue

            parts = line.split()
            if len(parts) < 9:
                continue

            sample_id = parts[0]
            status = parts[1]
            if status != "ok":
                skipped_status += 1
                continue

            text = " ".join(parts[8:]).replace("|", " ").strip()
            if not text:
                continue

            id_parts = sample_id.split("-")
            if len(id_parts) < 2:
                continue

            form1 = id_parts[0]
            form2 = f"{id_parts[0]}-{id_parts[1]}"

            if mode == "lines" or mode == "words":
                if len(id_parts) < 3:
                    continue
                img_path = os.path.join(img_dir, form1, form2, f"{sample_id}.png")
                split_key = form2
            elif mode == "forms":
                # Typical IAM form path: forms/a01/a01-000u.png
                img_path = os.path.join(img_dir, form1, f"{form2}.png")
                split_key = form2
            else:
                continue

            if not os.path.isfile(img_path):
                skipped_missing += 1
                continue

            rows.append({
                "image_path": img_path,
                "label": text,
                "split_key": split_key,
                "sample_id": sample_id,
            })

    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(f"No valid samples parsed for mode={mode}")

    print(f"[Data] Mode: {mode}")
    print(f"[Data] Parsed: {len(df):,}")
    print(f"[Data] Skipped non-ok: {skipped_status:,}")
    print(f"[Data] Skipped missing files: {skipped_missing:,}")
    return df

def parse_csv_layout(csv_path: str):
    df = pd.read_csv(csv_path)
    lc = {c.lower(): c for c in df.columns}

    image_col = lc.get("image_path") or lc.get("path") or lc.get("image")
    label_col = lc.get("label") or lc.get("text") or lc.get("transcript")

    if image_col is None or label_col is None:
        raise RuntimeError("CSV layout detected but required columns are missing")

    out = df[[image_col, label_col]].copy()
    out.columns = ["image_path", "label"]

    # Resolve relative paths from csv directory
    base = os.path.dirname(csv_path)
    resolved = []
    for p in out["image_path"].astype(str).tolist():
        if os.path.isabs(p) and os.path.isfile(p):
            resolved.append(p)
        else:
            rp = os.path.normpath(os.path.join(base, p))
            resolved.append(rp)
    out["image_path"] = resolved
    out["split_key"] = [f"row_{i}" for i in range(len(out))]
    out["sample_id"] = out["split_key"]
    out = out[out["label"].astype(str).str.len() > 0].reset_index(drop=True)
    out = out[out["image_path"].apply(os.path.isfile)].reset_index(drop=True)

    if out.empty:
        raise RuntimeError("CSV parsed but no valid image+label rows found")
    print(f"[Data] Mode: csv")
    print(f"[Data] Parsed: {len(out):,}")
    return out

def split_by_key(df: pd.DataFrame, seed: int = DEFAULT_SEED):
    keys = sorted(df["split_key"].unique())
    rng = random.Random(seed)
    rng.shuffle(keys)

    n = len(keys)
    n_train = int(0.8 * n)
    n_val = int(0.1 * n)

    train_keys = set(keys[:n_train])
    val_keys = set(keys[n_train:n_train + n_val])
    test_keys = set(keys[n_train + n_val:])

    train_df = df[df["split_key"].isin(train_keys)].reset_index(drop=True)
    val_df = df[df["split_key"].isin(val_keys)].reset_index(drop=True)
    test_df = df[df["split_key"].isin(test_keys)].reset_index(drop=True)

    print(f"[Split] Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}")
    return train_df, val_df, test_df

def encode_label(text: str, char_to_int: dict):
    out = []
    for ch in str(text):
        if ch in char_to_int:
            out.append(char_to_int[ch])
    return out

def preprocess_image(path: str, target_h: int, target_w: int):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None

    p1, p99 = np.percentile(img, 1), np.percentile(img, 99)
    if p99 - p1 > 5:
        img = np.clip((img.astype(np.float32) - p1) / (p99 - p1) * 255, 0, 255).astype(np.uint8)

    _, img = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    h, w = img.shape[:2]
    if h < 2 or w < 2:
        return None

    scale = target_h / h
    new_w = max(1, int(w * scale))
    resized = cv2.resize(img, (new_w, target_h), interpolation=cv2.INTER_AREA)

    if new_w >= target_w:
        out = resized[:, :target_w]
    else:
        pad = np.full((target_h, target_w - new_w), 255, dtype=np.uint8)
        out = np.concatenate([resized, pad], axis=1)

    out = (255.0 - out.astype(np.float32)) / 255.0
    out = np.expand_dims(out, axis=-1)
    return out.astype(np.float32)

class OCRSequence(tf.keras.utils.Sequence):
    def __init__(self, df: pd.DataFrame, char_to_int: dict, batch_size: int, max_label_len: int, img_h: int, img_w: int, augment: bool, shuffle: bool):
        self.df = df
        self.char_to_int = char_to_int
        self.batch_size = batch_size
        self.max_label_len = max_label_len
        self.img_h = img_h
        self.img_w = img_w
        self.augment = augment
        self.shuffle = shuffle
        self.indices = np.arange(len(df))
        self.on_epoch_end()

    def __len__(self):
        return max(1, len(self.indices) // self.batch_size)

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __getitem__(self, i):
        s = i * self.batch_size
        e = s + self.batch_size
        ids = self.indices[s:e]

        images = []
        labels = []

        for idx in ids:
            row = self.df.iloc[idx]
            img = preprocess_image(row["image_path"], self.img_h, self.img_w)
            if img is None:
                continue

            if self.augment and np.random.rand() < 0.35:
                noise = np.random.normal(0, 0.02, img.shape).astype(np.float32)
                img = np.clip(img + noise, 0.0, 1.0)

            enc = encode_label(row["label"], self.char_to_int)
            if not enc:
                continue

            images.append(img)
            labels.append(enc)

        if not images:
            dummy = np.zeros((self.img_h, self.img_w, 1), dtype=np.float32)
            images = [dummy]
            labels = [[0]]

        bsz = len(images)
        x = np.stack(images, axis=0)

        y = np.full((bsz, self.max_label_len), PAD_VALUE, dtype=np.int32)
        y_len = np.zeros((bsz,), dtype=np.int32)
        for j, enc in enumerate(labels):
            enc = enc[:self.max_label_len]
            y[j, :len(enc)] = enc
            y_len[j] = len(enc)

        input_len = np.full((bsz,), self.img_w // 4, dtype=np.int32)

        inputs = {
            "input_images": x,
            "label_encoded": y,
            "image_widths": input_len,
            "label_lengths": y_len,
        }
        return inputs, np.zeros((bsz,), dtype=np.float32)

class CTCLayer(layers.Layer):
    def call(self, inputs):
        y_true, y_pred, input_len, label_len = inputs
        if len(tf.shape(input_len)) == 1:
            input_len = tf.expand_dims(input_len, axis=-1)
        if len(tf.shape(label_len)) == 1:
            label_len = tf.expand_dims(label_len, axis=-1)
        loss = keras.backend.ctc_batch_cost(y_true, y_pred, input_len, label_len)
        self.add_loss(tf.reduce_mean(loss))
        return y_pred

def build_crnn(num_classes: int, img_h: int, img_w: int):
    in_img = layers.Input(shape=(img_h, img_w, 1), name="input_images", dtype="float32")
    in_lbl = layers.Input(shape=(None,), name="label_encoded", dtype="int32")
    in_w = layers.Input(shape=(), name="image_widths", dtype="int32")
    in_l = layers.Input(shape=(), name="label_lengths", dtype="int32")

    x = in_img
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(256, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 1))(x)

    shp = x.shape
    t_steps = int(shp[2])
    feat_dim = int(shp[1]) * int(shp[3])
    x = layers.Permute((2, 1, 3))(x)
    x = layers.Reshape((t_steps, feat_dim))(x)

    x = layers.Bidirectional(layers.LSTM(256, return_sequences=True))(x)
    x = layers.Dropout(0.25)(x)
    x = layers.Bidirectional(layers.LSTM(256, return_sequences=True))(x)
    x = layers.Dropout(0.25)(x)

    y_pred = layers.Dense(num_classes, activation="softmax", name="output_softmax")(x)
    out = CTCLayer(name="ctc_loss")([in_lbl, y_pred, in_w, in_l])

    train_model = keras.Model(inputs=[in_img, in_lbl, in_w, in_l], outputs=out, name="ocr_train")
    infer_model = keras.Model(inputs=in_img, outputs=y_pred, name="ocr_infer")
    return train_model, infer_model

@dataclass
class TrainConfig:
    epochs: int = 60
    batch_size: int = 16
    lr: float = 1e-3
    seed: int = DEFAULT_SEED
    output_dir: str = "/kaggle/working/fullpage_saved_models"

def run_training(cfg: TrainConfig):
    set_seed(cfg.seed)
    configure_gpu()

    layout = detect_dataset_layout("/kaggle/input")
    mode = layout["mode"]
    img_h = layout["img_h"]
    img_w = layout["img_w"]
    max_label_len = layout["max_label_len"]
    print(f"[Detect] Mode={mode}, image={img_h}x{img_w}, max_label_len={max_label_len}")

    if mode == "csv":
        df = parse_csv_layout(layout["csv"])
    else:
        df = parse_iam_txt_layout(layout["txt"], layout["img_dir"], mode)

    train_df, val_df, test_df = split_by_key(df, cfg.seed)

    char_to_int = {c: i for i, c in enumerate(ALPHABET)}
    num_classes = len(ALPHABET) + 1

    train_gen = OCRSequence(train_df, char_to_int, cfg.batch_size, max_label_len, img_h, img_w, augment=True, shuffle=True)
    val_gen = OCRSequence(val_df, char_to_int, cfg.batch_size, max_label_len, img_h, img_w, augment=False, shuffle=False)

    train_model, infer_model = build_crnn(num_classes=num_classes, img_h=img_h, img_w=img_w)
    train_model.compile(optimizer=keras.optimizers.Adam(learning_rate=cfg.lr))

    os.makedirs(cfg.output_dir, exist_ok=True)
    best_w = os.path.join(cfg.output_dir, "crnn_iam_best.weights.h5")
    final_w = os.path.join(cfg.output_dir, "crnn_iam_final.weights.h5")
    infer_p = os.path.join(cfg.output_dir, "crnn_iam_inference.keras")
    hist_p = os.path.join(cfg.output_dir, "training.csv")
    cmap_p = os.path.join(cfg.output_dir, "char_map.json")
    test_csv = os.path.join(cfg.output_dir, "test_split.csv")

    callbacks = [
        keras.callbacks.ModelCheckpoint(best_w, monitor="val_loss", save_best_only=True, save_weights_only=True, verbose=1),
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-6, verbose=1),
        keras.callbacks.CSVLogger(hist_p),
    ]

    print("=" * 80)
    print(f"Training OCR model (mode={mode})")
    print(f"Train samples: {len(train_df):,} | Val samples: {len(val_df):,}")
    print(f"Image size: {img_h}x{img_w} | Classes: {num_classes}")
    print(f"Epochs: {cfg.epochs} | Batch size: {cfg.batch_size} | LR: {cfg.lr}")
    print("=" * 80)

    train_model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=cfg.epochs,
        callbacks=callbacks,
        verbose=1,
    )

    train_model.save_weights(final_w)
    infer_model.save(infer_p)

    with open(cmap_p, "w", encoding="utf-8") as f:
        json.dump(
            {
                "alphabet": ALPHABET,
                "blank_index": BLANK_INDEX,
                "num_classes": num_classes,
                "image_height": img_h,
                "image_width": img_w,
                "max_label_len": max_label_len,
                "mode": mode,
            },
            f,
            indent=2,
        )

    test_df[["image_path", "label", "split_key", "sample_id"]].to_csv(test_csv, index=False)

    print("\nSaved outputs:")
    print(f"- {best_w}")
    print(f"- {final_w}")
    print(f"- {infer_p}")
    print(f"- {hist_p}")
    print(f"- {cmap_p}")
    print(f"- {test_csv}")

print("Pipeline loaded. Run the smoke-test cell next.")

In [ ]:
# Strong fallback patch: recursive archive extraction + generic annotation/image matching
import os
import tarfile
import zipfile
import pandas as pd

def _scan_tree(paths):
    files, dirs = [], []
    for base in paths:
        if not os.path.isdir(base):
            continue
        for root, dnames, fnames in os.walk(base):
            for d in dnames:
                dirs.append(os.path.join(root, d))
            for f in fnames:
                files.append(os.path.join(root, f))
    return files, dirs

def _extract_all_archives(search_roots, out_root='/kaggle/working/iam_auto_extracted'):
    os.makedirs(out_root, exist_ok=True)
    done = set()
    total = 0

    for _ in range(4):
        files, _ = _scan_tree(search_roots + [out_root])
        archives = [p for p in files if p.lower().endswith(('.zip', '.tar', '.tar.gz', '.tgz'))]
        new_count = 0
        for arc in archives:
            if arc in done:
                continue
            done.add(arc)
            base = os.path.basename(arc).replace('.tar.gz', '').replace('.tgz', '').replace('.tar', '').replace('.zip', '')
            target = os.path.join(out_root, base)
            os.makedirs(target, exist_ok=True)
            try:
                if arc.lower().endswith('.zip'):
                    with zipfile.ZipFile(arc, 'r') as zf:
                        zf.extractall(target)
                else:
                    with tarfile.open(arc, 'r:*') as tf:
                        tf.extractall(target)
                new_count += 1
                total += 1
            except Exception as e:
                print(f'[Patch] Skip archive {arc}: {e}')
        if new_count == 0:
            break

    print(f'[Patch] Total extracted archives: {total}')
    return out_root

def _build_image_index(search_roots):
    files, _ = _scan_tree(search_roots)
    img_files = [p for p in files if p.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp'))]
    index = {}
    for p in img_files:
        stem = os.path.splitext(os.path.basename(p))[0]
        if stem not in index:
            index[stem] = p
    print(f'[Patch] Indexed images: {len(index):,}')
    return index

def _find_annotation_txt(search_roots):
    files, _ = _scan_tree(search_roots)
    txts = [p for p in files if p.lower().endswith('.txt')]
    preferred = ['lines.txt', 'forms.txt', 'words.txt']
    ordered = []
    for name in preferred:
        ordered.extend([p for p in txts if os.path.basename(p).lower() == name])
    ordered.extend([p for p in txts if p not in ordered])

    def looks_like_iam_txt(path):
        try:
            with open(path, 'r', encoding='utf-8', errors='ignore') as f:
                checked = 0
                for raw in f:
                    line = raw.strip()
                    if not line or line.startswith('#'):
                        continue
                    parts = line.split()
                    if len(parts) >= 9 and '-' in parts[0]:
                        return True
                    checked += 1
                    if checked > 20:
                        break
        except Exception:
            return False
        return False

    for p in ordered:
        if looks_like_iam_txt(p):
            print(f'[Patch] Using annotation txt: {p}')
            return p
    raise FileNotFoundError('No IAM-like annotation txt found.')

def detect_dataset_layout(base='/kaggle/input'):
    extracted_root = _extract_all_archives([base])
    roots = [base, extracted_root]

    ann = _find_annotation_txt(roots)
    name = os.path.basename(ann).lower()
    if name == 'words.txt':
        mode, img_h, img_w, max_label_len = 'words', 64, 256, 48
    elif name == 'forms.txt':
        mode, img_h, img_w, max_label_len = 'forms', 64, 1536, 384
    else:
        mode, img_h, img_w, max_label_len = 'lines', 64, 1024, 128

    img_index = _build_image_index(roots)
    if not img_index:
        raise FileNotFoundError('No image files found after archive extraction.')

    return {
        'mode': mode,
        'txt': ann,
        'img_h': img_h,
        'img_w': img_w,
        'max_label_len': max_label_len,
        'img_index': img_index,
    }

def parse_iam_txt_layout(txt_path, img_index):
    rows = []
    skipped_status = 0
    skipped_missing = 0

    with open(txt_path, 'r', encoding='utf-8', errors='ignore') as f:
        for raw in f:
            line = raw.strip()
            if not line or line.startswith('#'):
                continue
            parts = line.split()
            if len(parts) < 9:
                continue

            sample_id = parts[0]
            status = parts[1]
            if status != 'ok':
                skipped_status += 1
                continue

            text = ' '.join(parts[8:]).replace('|', ' ').strip()
            if not text:
                continue

            img_path = img_index.get(sample_id)
            if img_path is None:
                skipped_missing += 1
                continue

            id_parts = sample_id.split('-')
            split_key = '-'.join(id_parts[:2]) if len(id_parts) >= 2 else sample_id
            rows.append({
                'image_path': img_path,
                'label': text,
                'split_key': split_key,
                'sample_id': sample_id,
            })

    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError('Parsed 0 samples from annotation file.')
    print(f"[Patch] Parsed samples: {len(df):,}")
    print(f"[Patch] Skipped non-ok: {skipped_status:,}")
    print(f"[Patch] Skipped missing-image ids: {skipped_missing:,}")
    return df

def run_training(cfg: TrainConfig):
    set_seed(cfg.seed)
    configure_gpu()

    layout = detect_dataset_layout('/kaggle/input')
    print(f"[Detect] Mode={layout['mode']}, image={layout['img_h']}x{layout['img_w']}, max_label_len={layout['max_label_len']}")

    df = parse_iam_txt_layout(layout['txt'], layout['img_index'])
    train_df, val_df, test_df = split_by_key(df, cfg.seed)

    char_to_int = {c: i for i, c in enumerate(ALPHABET)}
    num_classes = len(ALPHABET) + 1

    train_gen = OCRSequence(train_df, char_to_int, cfg.batch_size, layout['max_label_len'], layout['img_h'], layout['img_w'], augment=True, shuffle=True)
    val_gen = OCRSequence(val_df, char_to_int, cfg.batch_size, layout['max_label_len'], layout['img_h'], layout['img_w'], augment=False, shuffle=False)

    train_model, infer_model = build_crnn(num_classes=num_classes, img_h=layout['img_h'], img_w=layout['img_w'])
    train_model.compile(optimizer=keras.optimizers.Adam(learning_rate=cfg.lr))

    os.makedirs(cfg.output_dir, exist_ok=True)
    best_w = os.path.join(cfg.output_dir, 'crnn_iam_best.weights.h5')
    final_w = os.path.join(cfg.output_dir, 'crnn_iam_final.weights.h5')
    infer_p = os.path.join(cfg.output_dir, 'crnn_iam_inference.keras')
    hist_p = os.path.join(cfg.output_dir, 'training.csv')
    cmap_p = os.path.join(cfg.output_dir, 'char_map.json')
    test_csv = os.path.join(cfg.output_dir, 'test_split.csv')

    callbacks = [
        keras.callbacks.ModelCheckpoint(best_w, monitor='val_loss', save_best_only=True, save_weights_only=True, verbose=1),
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6, verbose=1),
        keras.callbacks.CSVLogger(hist_p),
    ]

    print('=' * 80)
    print(f"Training OCR model (mode={layout['mode']})")
    print(f"Train samples: {len(train_df):,} | Val samples: {len(val_df):,}")
    print(f"Image size: {layout['img_h']}x{layout['img_w']} | Classes: {num_classes}")
    print(f"Epochs: {cfg.epochs} | Batch size: {cfg.batch_size} | LR: {cfg.lr}")
    print('=' * 80)

    train_model.fit(train_gen, validation_data=val_gen, epochs=cfg.epochs, callbacks=callbacks, verbose=1)

    train_model.save_weights(final_w)
    infer_model.save(infer_p)

    with open(cmap_p, 'w', encoding='utf-8') as f:
        json.dump({
            'alphabet': ALPHABET,
            'blank_index': BLANK_INDEX,
            'num_classes': num_classes,
            'image_height': layout['img_h'],
            'image_width': layout['img_w'],
            'max_label_len': layout['max_label_len'],
            'mode': layout['mode'],
        }, f, indent=2)

    test_df[['image_path', 'label', 'split_key', 'sample_id']].to_csv(test_csv, index=False)

    print('\nSaved outputs:')
    print('-', best_w)
    print('-', final_w)
    print('-', infer_p)
    print('-', hist_p)
    print('-', cmap_p)
    print('-', test_csv)

print('Strong fallback patch loaded. Re-run smoke test cell now.')

In [ ]:
# Hotfix v2: format-agnostic annotation detection/parsing (run once after previous patch cell)
import os
import pandas as pd

def _probe_annotation_quality(path, max_lines=400):
    score = 0
    checked = 0
    try:
        with open(path, 'r', encoding='utf-8', errors='ignore') as f:
            for raw in f:
                line = raw.strip()
                if not line or line.startswith('#'):
                    continue
                checked += 1

                # IAM-style: id status ... text
                parts = line.split()
                if len(parts) >= 9 and '-' in parts[0]:
                    score += 3
                    continue

                # TSV/CSV/pipe style where first field is id/path and second+ is text
                for sep in ['\t', '|', ';', ',']:
                    if sep in line:
                        toks = [t.strip() for t in line.split(sep)]
                        if len(toks) >= 2 and toks[0] and any(toks[1:]):
                            score += 2
                            break
                else:
                    # Generic fallback: first token id/path + remaining free text
                    if len(parts) >= 2 and (('/' in parts[0]) or ('-' in parts[0]) or ('.' in parts[0])):
                        score += 1

                if checked >= max_lines:
                    break
    except Exception:
        return -1
    return score

def _find_annotation_txt(search_roots):
    files, _ = _scan_tree(search_roots)
    txts = [p for p in files if p.lower().endswith('.txt')]
    if not txts:
        raise FileNotFoundError('No .txt files found in dataset roots.')

    # Prefer likely annotation names first
    def _name_rank(p):
        n = os.path.basename(p).lower()
        if n in ('lines.txt', 'forms.txt', 'words.txt'):
            return 0
        if any(k in n for k in ['label', 'transcript', 'annotation', 'groundtruth', 'gt']):
            return 1
        return 2

    ordered = sorted(txts, key=lambda p: (_name_rank(p), len(p)))

    best_path = None
    best_score = -1
    for p in ordered:
        s = _probe_annotation_quality(p)
        if s > best_score:
            best_score = s
            best_path = p

    if best_path is None or best_score <= 0:
        print('[Patch-v2] Candidate txt files (first 40):')
        for p in ordered[:40]:
            print(' -', p)
        raise FileNotFoundError('No parseable annotation txt found.')

    print(f'[Patch-v2] Using annotation txt: {best_path} (score={best_score})')
    return best_path

def _extract_sample_and_text(line):
    parts = line.split()

    # IAM style
    if len(parts) >= 9 and '-' in parts[0]:
        sample_id = parts[0].strip()
        status = parts[1].strip().lower()
        if status not in ('ok', 'good', '1', 'true'):
            return None, None
        text = ' '.join(parts[8:]).replace('|', ' ').strip()
        return sample_id, text

    # Separator-based rows
    for sep in ['\t', '|', ';', ',']:
        if sep in line:
            toks = [t.strip() for t in line.split(sep)]
            if len(toks) >= 2 and toks[0]:
                sample_id = toks[0]
                text = ' '.join([t for t in toks[1:] if t]).strip()
                return sample_id, text

    # Generic fallback: first token = id/path, rest = label
    if len(parts) >= 2:
        sample_id = parts[0].strip()
        text = ' '.join(parts[1:]).strip()
        return sample_id, text

    return None, None

def _match_image_path(sample_id, img_index):
    if not sample_id:
        return None
    sid = str(sample_id).strip()

    candidates = [sid]
    candidates.append(os.path.basename(sid))
    candidates.append(os.path.splitext(os.path.basename(sid))[0])

    # IAM-like ids sometimes include directory prefixes; try progressively reduced forms
    if '/' in sid or '\\' in sid:
        bn = os.path.basename(sid)
        candidates.append(os.path.splitext(bn)[0])

    # If id looks like a path with extension, also try stem directly
    stem = os.path.splitext(sid)[0]
    candidates.append(stem)
    candidates.append(os.path.basename(stem))

    seen = set()
    for c in candidates:
        if not c or c in seen:
            continue
        seen.add(c)
        if c in img_index:
            return img_index[c]
    return None

def parse_iam_txt_layout(txt_path, img_index):
    rows = []
    skipped_missing = 0
    skipped_bad = 0

    with open(txt_path, 'r', encoding='utf-8', errors='ignore') as f:
        for raw in f:
            line = raw.strip()
            if not line or line.startswith('#'):
                continue

            sample_id, text = _extract_sample_and_text(line)
            if not sample_id or not text:
                skipped_bad += 1
                continue

            img_path = _match_image_path(sample_id, img_index)
            if img_path is None:
                skipped_missing += 1
                continue

            sid = os.path.splitext(os.path.basename(str(sample_id)))[0]
            id_parts = sid.split('-')
            split_key = '-'.join(id_parts[:2]) if len(id_parts) >= 2 else sid

            rows.append({
                'image_path': img_path,
                'label': text.replace('|', ' ').strip(),
                'split_key': split_key,
                'sample_id': sid,
            })

    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError('Parsed 0 samples from annotation file (v2 parser).')

    print(f"[Patch-v2] Parsed samples: {len(df):,}")
    print(f"[Patch-v2] Skipped malformed rows: {skipped_bad:,}")
    print(f"[Patch-v2] Skipped missing-image rows: {skipped_missing:,}")
    return df

print('Hotfix v2 loaded. Run Smoke Test cell now.')

In [ ]:
# Hotfix v3: resilient source discovery + stronger image id matching + run_training fallback
import os
import re
import pandas as pd

def _norm_key(s):
    s = str(s).strip().lower()
    s = os.path.splitext(os.path.basename(s))[0]
    return re.sub(r'[^a-z0-9]+', '', s)

def _build_image_index(search_roots):
    files, _ = _scan_tree(search_roots)
    img_files = [p for p in files if p.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp'))]
    index = {}
    norm_index = {}
    for p in img_files:
        b = os.path.basename(p)
        stem = os.path.splitext(b)[0]
        for k in [stem, stem.lower(), b, b.lower()]:
            if k not in index:
                index[k] = p
        nk = _norm_key(stem)
        if nk and nk not in norm_index:
            norm_index[nk] = p
    print(f'[Patch-v3] Indexed images: {len(img_files):,} files, {len(index):,} exact keys, {len(norm_index):,} normalized keys')
    return {'exact': index, 'norm': norm_index}

def _find_annotation_source(search_roots):
    files, _ = _scan_tree(search_roots)
    txts = [p for p in files if p.lower().endswith('.txt')]
    csvs = [p for p in files if p.lower().endswith(('.csv', '.tsv'))]

    best_txt = None
    best_score = -1
    for p in txts:
        s = _probe_annotation_quality(p) if '_probe_annotation_quality' in globals() else 0
        if s > best_score:
            best_score = s
            best_txt = p

    if best_txt and best_score > 0:
        print(f'[Patch-v3] Selected txt annotation: {best_txt} (score={best_score})')
        return {'kind': 'txt', 'path': best_txt}

    # Fallback: scan csv/tsv for likely columns
    for p in sorted(csvs):
        try:
            if p.lower().endswith('.tsv'):
                dfh = pd.read_csv(p, sep='\t', nrows=5)
            else:
                dfh = pd.read_csv(p, nrows=5)
            lc = {c.lower() for c in dfh.columns}
            has_img = any(c in lc for c in ['image_path', 'path', 'image', 'file', 'filename'])
            has_lbl = any(c in lc for c in ['label', 'text', 'transcript', 'gt', 'ground_truth'])
            if has_img and has_lbl:
                print(f'[Patch-v3] Selected tabular annotation: {p}')
                return {'kind': 'csv', 'path': p}
        except Exception:
            continue

    print('[Patch-v3] Annotation candidates (first 40):')
    for p in sorted([x for x in files if x.lower().endswith(('.txt', '.csv', '.tsv', '.json', '.xml'))])[:40]:
        print(' -', p)
    raise FileNotFoundError('No parseable annotation source found (txt/csv/tsv).')

def _match_image_path(sample_id, img_index):
    sid = str(sample_id).strip()
    candidates = [
        sid, sid.lower(),
        os.path.basename(sid), os.path.basename(sid).lower(),
        os.path.splitext(os.path.basename(sid))[0], os.path.splitext(os.path.basename(sid))[0].lower(),
        os.path.splitext(sid)[0], os.path.splitext(sid)[0].lower(),
    ]
    seen = set()
    for c in candidates:
        if not c or c in seen:
            continue
        seen.add(c)
        p = img_index['exact'].get(c)
        if p is not None:
            return p

    nk = _norm_key(sid)
    if nk in img_index['norm']:
        return img_index['norm'][nk]

    # Last-resort prefix/suffix match on normalized ids
    if nk:
        for k, p in img_index['norm'].items():
            if nk in k or k in nk:
                return p
    return None

def detect_dataset_layout(base='/kaggle/input'):
    extracted_root = _extract_all_archives([base]) if '_extract_all_archives' in globals() else base
    roots = [base, extracted_root]
    ann = _find_annotation_source(roots)
    img_index = _build_image_index(roots)
    if not img_index['exact'] and not img_index['norm']:
        raise FileNotFoundError('No image files found after archive extraction.')

    mode, img_h, img_w, max_label_len = 'lines', 64, 1024, 128
    if ann['kind'] == 'txt':
        n = os.path.basename(ann['path']).lower()
        if n == 'words.txt':
            mode, img_h, img_w, max_label_len = 'words', 64, 256, 48
        elif n == 'forms.txt':
            mode, img_h, img_w, max_label_len = 'forms', 64, 1536, 384

    return {
        'mode': mode,
        'ann_kind': ann['kind'],
        'txt': ann['path'] if ann['kind'] == 'txt' else None,
        'csv': ann['path'] if ann['kind'] == 'csv' else None,
        'img_h': img_h,
        'img_w': img_w,
        'max_label_len': max_label_len,
        'img_index': img_index,
    }

def parse_iam_txt_layout(txt_path, img_index):
    rows = []
    skipped_missing = 0
    skipped_bad = 0

    with open(txt_path, 'r', encoding='utf-8', errors='ignore') as f:
        for raw in f:
            line = raw.strip()
            if not line or line.startswith('#'):
                continue

            sample_id, text = _extract_sample_and_text(line) if '_extract_sample_and_text' in globals() else (None, None)
            if not sample_id or not text:
                parts = line.split()
                if len(parts) >= 2:
                    sample_id = parts[0].strip()
                    text = ' '.join(parts[1:]).strip()
                else:
                    skipped_bad += 1
                    continue

            img_path = _match_image_path(sample_id, img_index)
            if img_path is None:
                skipped_missing += 1
                continue

            sid = os.path.splitext(os.path.basename(str(sample_id)))[0]
            sid = sid if sid else str(sample_id).strip()
            id_parts = sid.split('-')
            split_key = '-'.join(id_parts[:2]) if len(id_parts) >= 2 else sid

            rows.append({
                'image_path': img_path,
                'label': str(text).replace('|', ' ').strip(),
                'split_key': split_key,
                'sample_id': sid,
            })

    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError('Parsed 0 samples from annotation file (v3 parser).')
    print(f"[Patch-v3] Parsed samples: {len(df):,}")
    print(f"[Patch-v3] Skipped malformed rows: {skipped_bad:,}")
    print(f"[Patch-v3] Skipped missing-image rows: {skipped_missing:,}")
    return df

def run_training(cfg: TrainConfig):
    set_seed(cfg.seed)
    configure_gpu()

    layout = detect_dataset_layout('/kaggle/input')
    print(f"[Detect-v3] Mode={layout['mode']}, ann_kind={layout['ann_kind']}, image={layout['img_h']}x{layout['img_w']}, max_label_len={layout['max_label_len']}")

    if layout['ann_kind'] == 'csv':
        df = parse_csv_layout(layout['csv'])
    else:
        df = parse_iam_txt_layout(layout['txt'], layout['img_index'])

    train_df, val_df, test_df = split_by_key(df, cfg.seed)

    char_to_int = {c: i for i, c in enumerate(ALPHABET)}
    num_classes = len(ALPHABET) + 1

    train_gen = OCRSequence(train_df, char_to_int, cfg.batch_size, layout['max_label_len'], layout['img_h'], layout['img_w'], augment=True, shuffle=True)
    val_gen = OCRSequence(val_df, char_to_int, cfg.batch_size, layout['max_label_len'], layout['img_h'], layout['img_w'], augment=False, shuffle=False)

    train_model, infer_model = build_crnn(num_classes=num_classes, img_h=layout['img_h'], img_w=layout['img_w'])
    train_model.compile(optimizer=keras.optimizers.Adam(learning_rate=cfg.lr))

    os.makedirs(cfg.output_dir, exist_ok=True)
    best_w = os.path.join(cfg.output_dir, 'crnn_iam_best.weights.h5')
    final_w = os.path.join(cfg.output_dir, 'crnn_iam_final.weights.h5')
    infer_p = os.path.join(cfg.output_dir, 'crnn_iam_inference.keras')
    hist_p = os.path.join(cfg.output_dir, 'training.csv')
    cmap_p = os.path.join(cfg.output_dir, 'char_map.json')
    test_csv = os.path.join(cfg.output_dir, 'test_split.csv')

    callbacks = [
        keras.callbacks.ModelCheckpoint(best_w, monitor='val_loss', save_best_only=True, save_weights_only=True, verbose=1),
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6, verbose=1),
        keras.callbacks.CSVLogger(hist_p),
    ]

    print('=' * 80)
    print(f"Training OCR model (mode={layout['mode']})")
    print(f"Train samples: {len(train_df):,} | Val samples: {len(val_df):,}")
    print(f"Image size: {layout['img_h']}x{layout['img_w']} | Classes: {num_classes}")
    print(f"Epochs: {cfg.epochs} | Batch size: {cfg.batch_size} | LR: {cfg.lr}")
    print('=' * 80)

    train_model.fit(train_gen, validation_data=val_gen, epochs=cfg.epochs, callbacks=callbacks, verbose=1)

    train_model.save_weights(final_w)
    infer_model.save(infer_p)

    with open(cmap_p, 'w', encoding='utf-8') as f:
        json.dump({
            'alphabet': ALPHABET,
            'blank_index': BLANK_INDEX,
            'num_classes': num_classes,
            'image_height': layout['img_h'],
            'image_width': layout['img_w'],
            'max_label_len': layout['max_label_len'],
            'mode': layout['mode'],
        }, f, indent=2)

    test_df[['image_path', 'label', 'split_key', 'sample_id']].to_csv(test_csv, index=False)

    print('\nSaved outputs:')
    print('-', best_w)
    print('-', final_w)
    print('-', infer_p)
    print('-', hist_p)
    print('-', cmap_p)
    print('-', test_csv)

print('Hotfix v3 loaded. Run Smoke Test cell now.')

In [ ]:
# Hotfix v4: JSON annotation support (__output__.json and similar)
import json
import os
import pandas as pd

def _flatten_dict_records(obj):
    records = []
    if isinstance(obj, dict):
        records.append(obj)
        for v in obj.values():
            records.extend(_flatten_dict_records(v))
    elif isinstance(obj, list):
        for x in obj:
            records.extend(_flatten_dict_records(x))
    return records

def _find_annotation_source(search_roots):
    files, _ = _scan_tree(search_roots)
    txts = [p for p in files if p.lower().endswith('.txt')]
    csvs = [p for p in files if p.lower().endswith(('.csv', '.tsv'))]
    jsons = [p for p in files if p.lower().endswith('.json')]

    best_txt = None
    best_score = -1
    for p in txts:
        s = _probe_annotation_quality(p) if '_probe_annotation_quality' in globals() else 0
        if s > best_score:
            best_score = s
            best_txt = p
    if best_txt and best_score > 0:
        print(f'[Patch-v4] Selected txt annotation: {best_txt} (score={best_score})')
        return {'kind': 'txt', 'path': best_txt}

    for p in sorted(csvs):
        try:
            if p.lower().endswith('.tsv'):
                dfh = pd.read_csv(p, sep='\t', nrows=5)
            else:
                dfh = pd.read_csv(p, nrows=5)
            lc = {c.lower() for c in dfh.columns}
            has_img = any(c in lc for c in ['image_path', 'path', 'image', 'file', 'filename'])
            has_lbl = any(c in lc for c in ['label', 'text', 'transcript', 'gt', 'ground_truth'])
            if has_img and has_lbl:
                print(f'[Patch-v4] Selected tabular annotation: {p}')
                return {'kind': 'csv', 'path': p}
        except Exception:
            continue

    # JSON fallback: prefer __output__.json then any json containing record-like dicts
    ranked_jsons = sorted(jsons, key=lambda p: (0 if os.path.basename(p).lower() == '__output__.json' else 1, len(p)))
    for p in ranked_jsons:
        try:
            with open(p, 'r', encoding='utf-8', errors='ignore') as f:
                obj = json.load(f)
            recs = _flatten_dict_records(obj)
            if not recs:
                continue
            # quick probe: check if at least one record has both image-like and label-like fields
            img_keys = {'image_path', 'path', 'image', 'img', 'img_path', 'file', 'filename', 'name', 'id'}
            lbl_keys = {'label', 'text', 'transcript', 'gt', 'ground_truth', 'word', 'line', 'sentence', 'content'}
            for r in recs[:2000]:
                keys = {str(k).lower() for k in r.keys()}
                if keys & img_keys and keys & lbl_keys:
                    print(f'[Patch-v4] Selected JSON annotation: {p}')
                    return {'kind': 'json', 'path': p}
            # also allow filename->label dict mappings
            if isinstance(obj, dict) and obj:
                sample_k = next(iter(obj.keys()))
                sample_v = obj[sample_k]
                if isinstance(sample_k, str) and isinstance(sample_v, str):
                    print(f'[Patch-v4] Selected JSON key/value annotation: {p}')
                    return {'kind': 'json', 'path': p}
        except Exception:
            continue

    print('[Patch-v4] Annotation candidates (first 40):')
    for p in sorted([x for x in files if x.lower().endswith(('.txt', '.csv', '.tsv', '.json', '.xml'))])[:40]:
        print(' -', p)
    raise FileNotFoundError('No parseable annotation source found (txt/csv/tsv/json).')

def parse_json_layout(json_path, img_index):
    with open(json_path, 'r', encoding='utf-8', errors='ignore') as f:
        obj = json.load(f)

    rows = []
    missing = 0
    bad = 0

    def _pick(d, candidates):
        for c in candidates:
            if c in d and d[c] is not None and str(d[c]).strip() != '':
                return d[c]
        return None

    # Case 1: direct filename->label mapping
    if isinstance(obj, dict) and obj and all(isinstance(k, str) for k in obj.keys()) and all(isinstance(v, str) for v in obj.values()):
        for k, v in obj.items():
            img_path = _match_image_path(k, img_index)
            if img_path is None:
                missing += 1
                continue
            sid = os.path.splitext(os.path.basename(str(k)))[0]
            id_parts = sid.split('-')
            split_key = '-'.join(id_parts[:2]) if len(id_parts) >= 2 else sid
            rows.append({'image_path': img_path, 'label': str(v).strip(), 'split_key': split_key, 'sample_id': sid})
    else:
        recs = _flatten_dict_records(obj)
        for r in recs:
            if not isinstance(r, dict):
                continue
            d = {str(k).lower(): v for k, v in r.items()}
            sample = _pick(d, ['image_path', 'path', 'image', 'img', 'img_path', 'file', 'filename', 'name', 'id'])
            text = _pick(d, ['label', 'text', 'transcript', 'gt', 'ground_truth', 'word', 'line', 'sentence', 'content'])
            if sample is None or text is None:
                continue
            sample = str(sample).strip()
            text = str(text).strip()
            if not sample or not text:
                bad += 1
                continue
            img_path = _match_image_path(sample, img_index)
            if img_path is None:
                missing += 1
                continue
            sid = os.path.splitext(os.path.basename(sample))[0]
            id_parts = sid.split('-')
            split_key = '-'.join(id_parts[:2]) if len(id_parts) >= 2 else sid
            rows.append({'image_path': img_path, 'label': text, 'split_key': split_key, 'sample_id': sid})

    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(f'JSON parsed but produced 0 usable rows: {json_path}')

    # sanitize labels
    df['label'] = df['label'].astype(str).str.replace('|', ' ', regex=False).str.strip()
    df = df[df['label'].str.len() > 0].reset_index(drop=True)
    if df.empty:
        raise RuntimeError(f'JSON rows found but labels are empty after sanitize: {json_path}')

    print(f"[Patch-v4] JSON parsed samples: {len(df):,}")
    print(f"[Patch-v4] JSON missing-image rows: {missing:,}")
    print(f"[Patch-v4] JSON malformed rows: {bad:,}")
    return df

def detect_dataset_layout(base='/kaggle/input'):
    extracted_root = _extract_all_archives([base]) if '_extract_all_archives' in globals() else base
    roots = [base, extracted_root]
    ann = _find_annotation_source(roots)
    img_index = _build_image_index(roots)
    if not img_index['exact'] and not img_index['norm']:
        raise FileNotFoundError('No image files found after archive extraction.')

    mode, img_h, img_w, max_label_len = 'lines', 64, 1024, 128
    if ann['kind'] == 'txt':
        n = os.path.basename(ann['path']).lower()
        if n == 'words.txt':
            mode, img_h, img_w, max_label_len = 'words', 64, 256, 48
        elif n == 'forms.txt':
            mode, img_h, img_w, max_label_len = 'forms', 64, 1536, 384

    return {
        'mode': mode,
        'ann_kind': ann['kind'],
        'txt': ann['path'] if ann['kind'] == 'txt' else None,
        'csv': ann['path'] if ann['kind'] == 'csv' else None,
        'json': ann['path'] if ann['kind'] == 'json' else None,
        'img_h': img_h,
        'img_w': img_w,
        'max_label_len': max_label_len,
        'img_index': img_index,
    }

def run_training(cfg: TrainConfig):
    set_seed(cfg.seed)
    configure_gpu()

    layout = detect_dataset_layout('/kaggle/input')
    print(f"[Detect-v4] Mode={layout['mode']}, ann_kind={layout['ann_kind']}, image={layout['img_h']}x{layout['img_w']}, max_label_len={layout['max_label_len']}")

    if layout['ann_kind'] == 'csv':
        df = parse_csv_layout(layout['csv'])
    elif layout['ann_kind'] == 'json':
        df = parse_json_layout(layout['json'], layout['img_index'])
    else:
        df = parse_iam_txt_layout(layout['txt'], layout['img_index'])

    train_df, val_df, test_df = split_by_key(df, cfg.seed)

    char_to_int = {c: i for i, c in enumerate(ALPHABET)}
    num_classes = len(ALPHABET) + 1

    train_gen = OCRSequence(train_df, char_to_int, cfg.batch_size, layout['max_label_len'], layout['img_h'], layout['img_w'], augment=True, shuffle=True)
    val_gen = OCRSequence(val_df, char_to_int, cfg.batch_size, layout['max_label_len'], layout['img_h'], layout['img_w'], augment=False, shuffle=False)

    train_model, infer_model = build_crnn(num_classes=num_classes, img_h=layout['img_h'], img_w=layout['img_w'])
    train_model.compile(optimizer=keras.optimizers.Adam(learning_rate=cfg.lr))

    os.makedirs(cfg.output_dir, exist_ok=True)
    best_w = os.path.join(cfg.output_dir, 'crnn_iam_best.weights.h5')
    final_w = os.path.join(cfg.output_dir, 'crnn_iam_final.weights.h5')
    infer_p = os.path.join(cfg.output_dir, 'crnn_iam_inference.keras')
    hist_p = os.path.join(cfg.output_dir, 'training.csv')
    cmap_p = os.path.join(cfg.output_dir, 'char_map.json')
    test_csv = os.path.join(cfg.output_dir, 'test_split.csv')

    callbacks = [
        keras.callbacks.ModelCheckpoint(best_w, monitor='val_loss', save_best_only=True, save_weights_only=True, verbose=1),
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6, verbose=1),
        keras.callbacks.CSVLogger(hist_p),
    ]

    print('=' * 80)
    print(f"Training OCR model (mode={layout['mode']})")
    print(f"Train samples: {len(train_df):,} | Val samples: {len(val_df):,}")
    print(f"Image size: {layout['img_h']}x{layout['img_w']} | Classes: {num_classes}")
    print(f"Epochs: {cfg.epochs} | Batch size: {cfg.batch_size} | LR: {cfg.lr}")
    print('=' * 80)

    train_model.fit(train_gen, validation_data=val_gen, epochs=cfg.epochs, callbacks=callbacks, verbose=1)

    train_model.save_weights(final_w)
    infer_model.save(infer_p)

    with open(cmap_p, 'w', encoding='utf-8') as f:
        json.dump({
            'alphabet': ALPHABET,
            'blank_index': BLANK_INDEX,
            'num_classes': num_classes,
            'image_height': layout['img_h'],
            'image_width': layout['img_w'],
            'max_label_len': layout['max_label_len'],
            'mode': layout['mode'],
            'ann_kind': layout['ann_kind'],
        }, f, indent=2)

    test_df[['image_path', 'label', 'split_key', 'sample_id']].to_csv(test_csv, index=False)

    print('\nSaved outputs:')
    print('-', best_w)
    print('-', final_w)
    print('-', infer_p)
    print('-', hist_p)
    print('-', cmap_p)
    print('-', test_csv)

print('Hotfix v4 loaded. Now run Smoke Test again.')

In [ ]:
# Hotfix v5: force JSON fallback + deep nested JSON parser
import json
import os
import pandas as pd

def _find_annotation_source(search_roots):
    files, _ = _scan_tree(search_roots)
    txts = [p for p in files if p.lower().endswith('.txt')]
    csvs = [p for p in files if p.lower().endswith(('.csv', '.tsv'))]
    jsons = [p for p in files if p.lower().endswith('.json')]

    best_txt = None
    best_score = -1
    for p in txts:
        s = _probe_annotation_quality(p) if '_probe_annotation_quality' in globals() else 0
        if s > best_score:
            best_score = s
            best_txt = p
    if best_txt and best_score > 0:
        print(f'[Patch-v5] Selected txt annotation: {best_txt} (score={best_score})')
        return {'kind': 'txt', 'path': best_txt}

    for p in sorted(csvs):
        try:
            dfh = pd.read_csv(p, sep='\t' if p.lower().endswith('.tsv') else ',', nrows=5)
            lc = {c.lower() for c in dfh.columns}
            has_img = any(c in lc for c in ['image_path', 'path', 'image', 'file', 'filename'])
            has_lbl = any(c in lc for c in ['label', 'text', 'transcript', 'gt', 'ground_truth'])
            if has_img and has_lbl:
                print(f'[Patch-v5] Selected tabular annotation: {p}')
                return {'kind': 'csv', 'path': p}
        except Exception:
            continue

    if jsons:
        ranked = sorted(jsons, key=lambda p: (0 if os.path.basename(p).lower() == '__output__.json' else 1, len(p)))
        pick = ranked[0]
        print(f'[Patch-v5] Selected JSON annotation by fallback: {pick}')
        return {'kind': 'json', 'path': pick}

    print('[Patch-v5] Annotation candidates (first 40):')
    for p in sorted([x for x in files if x.lower().endswith(('.txt', '.csv', '.tsv', '.json', '.xml'))])[:40]:
        print(' -', p)
    raise FileNotFoundError('No parseable annotation source found (txt/csv/tsv/json).')

def _extract_text_value(v):
    if v is None:
        return None
    if isinstance(v, str):
        t = v.strip()
        return t if t else None
    if isinstance(v, list):
        parts = [str(x).strip() for x in v if str(x).strip()]
        if parts:
            return ' '.join(parts)
    return None

def _pick_img_key(d):
    for k in ['image_path', 'path', 'image', 'img', 'img_path', 'file', 'filename', 'name', 'id']:
        if k in d and str(d[k]).strip():
            return str(d[k]).strip()
    return None

def _pick_label_key(d):
    for k in ['label', 'text', 'transcript', 'gt', 'ground_truth', 'word', 'line', 'sentence', 'content']:
        if k in d:
            t = _extract_text_value(d[k])
            if t:
                return t
    return None

def _collect_pairs(node, current_img=None, out=None):
    if out is None:
        out = []

    if isinstance(node, dict):
        d = {str(k).lower(): v for k, v in node.items()}

        # Label Studio common path: data.image + annotations[].result[].value.text
        if 'data' in d and isinstance(d['data'], dict):
            data_low = {str(k).lower(): v for k, v in d['data'].items()}
            ls_img = _pick_img_key(data_low)
            if ls_img:
                current_img = ls_img

        img_here = _pick_img_key(d) or current_img
        lbl_here = _pick_label_key(d)
        if img_here and lbl_here:
            out.append((img_here, lbl_here))

        # Special handling for value.text arrays
        if 'value' in d and isinstance(d['value'], dict):
            vlow = {str(k).lower(): v for k, v in d['value'].items()}
            txt = _pick_label_key(vlow)
            if txt and current_img:
                out.append((current_img, txt))

        for v in d.values():
            _collect_pairs(v, current_img=img_here, out=out)

    elif isinstance(node, list):
        for item in node:
            _collect_pairs(item, current_img=current_img, out=out)

    return out

def parse_json_layout(json_path, img_index):
    try:
        with open(json_path, 'r', encoding='utf-8', errors='ignore') as f:
            obj = json.load(f)
    except Exception as e:
        raise RuntimeError(f'JSON load failed for {json_path}: {e}')

    pairs = _collect_pairs(obj, current_img=None, out=[])
    if not pairs:
        # last resort: if dict mapping filename->text
        if isinstance(obj, dict):
            for k, v in obj.items():
                if isinstance(k, str):
                    t = _extract_text_value(v)
                    if t:
                        pairs.append((k, t))

    rows = []
    missing = 0
    bad = 0
    seen = set()
    for sample, text in pairs:
        if not sample or not text:
            bad += 1
            continue
        img_path = _match_image_path(sample, img_index)
        if img_path is None:
            missing += 1
            continue
        sid = os.path.splitext(os.path.basename(str(sample)))[0]
        id_parts = sid.split('-')
        split_key = '-'.join(id_parts[:2]) if len(id_parts) >= 2 else sid
        key = (img_path, text)
        if key in seen:
            continue
        seen.add(key)
        rows.append({'image_path': img_path, 'label': str(text).replace('|', ' ').strip(), 'split_key': split_key, 'sample_id': sid})

    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(f'JSON parsed but produced 0 usable rows: {json_path}')

    df = df[df['label'].astype(str).str.len() > 0].reset_index(drop=True)
    if df.empty:
        raise RuntimeError(f'JSON rows found but labels are empty after sanitize: {json_path}')

    print(f"[Patch-v5] JSON pair candidates: {len(pairs):,}")
    print(f"[Patch-v5] JSON parsed samples: {len(df):,}")
    print(f"[Patch-v5] JSON missing-image rows: {missing:,}")
    print(f"[Patch-v5] JSON malformed rows: {bad:,}")
    return df

def detect_dataset_layout(base='/kaggle/input'):
    extracted_root = _extract_all_archives([base]) if '_extract_all_archives' in globals() else base
    roots = [base, extracted_root]
    ann = _find_annotation_source(roots)
    img_index = _build_image_index(roots)
    if not img_index['exact'] and not img_index['norm']:
        raise FileNotFoundError('No image files found after archive extraction.')

    mode, img_h, img_w, max_label_len = 'lines', 64, 1024, 128
    if ann['kind'] == 'txt':
        n = os.path.basename(ann['path']).lower()
        if n == 'words.txt':
            mode, img_h, img_w, max_label_len = 'words', 64, 256, 48
        elif n == 'forms.txt':
            mode, img_h, img_w, max_label_len = 'forms', 64, 1536, 384

    return {
        'mode': mode,
        'ann_kind': ann['kind'],
        'txt': ann['path'] if ann['kind'] == 'txt' else None,
        'csv': ann['path'] if ann['kind'] == 'csv' else None,
        'json': ann['path'] if ann['kind'] == 'json' else None,
        'img_h': img_h,
        'img_w': img_w,
        'max_label_len': max_label_len,
        'img_index': img_index,
    }

def run_training(cfg: TrainConfig):
    set_seed(cfg.seed)
    configure_gpu()

    layout = detect_dataset_layout('/kaggle/input')
    print(f"[Detect-v5] Mode={layout['mode']}, ann_kind={layout['ann_kind']}, image={layout['img_h']}x{layout['img_w']}, max_label_len={layout['max_label_len']}")

    if layout['ann_kind'] == 'csv':
        df = parse_csv_layout(layout['csv'])
    elif layout['ann_kind'] == 'json':
        df = parse_json_layout(layout['json'], layout['img_index'])
    else:
        df = parse_iam_txt_layout(layout['txt'], layout['img_index'])

    train_df, val_df, test_df = split_by_key(df, cfg.seed)

    char_to_int = {c: i for i, c in enumerate(ALPHABET)}
    num_classes = len(ALPHABET) + 1

    train_gen = OCRSequence(train_df, char_to_int, cfg.batch_size, layout['max_label_len'], layout['img_h'], layout['img_w'], augment=True, shuffle=True)
    val_gen = OCRSequence(val_df, char_to_int, cfg.batch_size, layout['max_label_len'], layout['img_h'], layout['img_w'], augment=False, shuffle=False)

    train_model, infer_model = build_crnn(num_classes=num_classes, img_h=layout['img_h'], img_w=layout['img_w'])
    train_model.compile(optimizer=keras.optimizers.Adam(learning_rate=cfg.lr))

    os.makedirs(cfg.output_dir, exist_ok=True)
    best_w = os.path.join(cfg.output_dir, 'crnn_iam_best.weights.h5')
    final_w = os.path.join(cfg.output_dir, 'crnn_iam_final.weights.h5')
    infer_p = os.path.join(cfg.output_dir, 'crnn_iam_inference.keras')
    hist_p = os.path.join(cfg.output_dir, 'training.csv')
    cmap_p = os.path.join(cfg.output_dir, 'char_map.json')
    test_csv = os.path.join(cfg.output_dir, 'test_split.csv')

    callbacks = [
        keras.callbacks.ModelCheckpoint(best_w, monitor='val_loss', save_best_only=True, save_weights_only=True, verbose=1),
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6, verbose=1),
        keras.callbacks.CSVLogger(hist_p),
    ]

    print('=' * 80)
    print(f"Training OCR model (mode={layout['mode']})")
    print(f"Train samples: {len(train_df):,} | Val samples: {len(val_df):,}")
    print(f"Image size: {layout['img_h']}x{layout['img_w']} | Classes: {num_classes}")
    print(f"Epochs: {cfg.epochs} | Batch size: {cfg.batch_size} | LR: {cfg.lr}")
    print('=' * 80)

    train_model.fit(train_gen, validation_data=val_gen, epochs=cfg.epochs, callbacks=callbacks, verbose=1)

    train_model.save_weights(final_w)
    infer_model.save(infer_p)

    with open(cmap_p, 'w', encoding='utf-8') as f:
        json.dump({
            'alphabet': ALPHABET,
            'blank_index': BLANK_INDEX,
            'num_classes': num_classes,
            'image_height': layout['img_h'],
            'image_width': layout['img_w'],
            'max_label_len': layout['max_label_len'],
            'mode': layout['mode'],
            'ann_kind': layout['ann_kind'],
        }, f, indent=2)

    test_df[['image_path', 'label', 'split_key', 'sample_id']].to_csv(test_csv, index=False)

    print('\nSaved outputs:')
    print('-', best_w)
    print('-', final_w)
    print('-', infer_p)
    print('-', hist_p)
    print('-', cmap_p)
    print('-', test_csv)

print('Hotfix v5 loaded. Run Smoke Test again.')

## 2b) Dataset Detection Fix (Archive-Aware)
Run this once after Cell 6. It auto-extracts IAM archives if needed and patches dataset detection.

## 3) Smoke Test (Quick)
Run this first to ensure IAM detection and training loop are working.

In [ ]:
# Smoke test run (quick) with preflight diagnostics
import os
import traceback

def _list_candidates(root='/kaggle/input', max_items=120):
    out = []
    for base, _, files in os.walk(root):
        for f in files:
            lp = os.path.join(base, f)
            if f.lower().endswith(('.txt', '.csv', '.tsv', '.json', '.xml', '.zip', '.tar', '.tar.gz', '.tgz')):
                out.append(lp)
                if len(out) >= max_items:
                    return out
    return out

print('[Smoke] Starting preflight...')
print('[Smoke] /kaggle/input entries:', sorted(os.listdir('/kaggle/input')) if os.path.isdir('/kaggle/input') else 'MISSING')

cands = _list_candidates('/kaggle/input', max_items=80)
print(f'[Smoke] Candidate annotation/archive files: {len(cands)}')
for p in cands[:40]:
    print(' -', p)

try:
    cfg = TrainConfig(epochs=1, batch_size=4, lr=1e-3)
    run_training(cfg)
    print('[Smoke] SUCCESS')
except Exception as e:
    print('\n[Smoke] FAILED:', repr(e))
    print('[Smoke] Traceback:')
    traceback.print_exc(limit=10)

    # Additional focused diagnostics after failure
    try:
        roots = ['/kaggle/input']
        if '_extract_all_archives' in globals():
            ex = _extract_all_archives(['/kaggle/input'])
            roots.append(ex)
            print('[Smoke] Extracted root:', ex)
        files, _ = _scan_tree(roots) if '_scan_tree' in globals() else ([], [])
        print(f'[Smoke] Total files scanned after failure: {len(files):,}')

        txt_like = [p for p in files if p.lower().endswith(('.txt', '.csv', '.tsv', '.json', '.xml'))]
        img_like = [p for p in files if p.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp'))]
        print(f'[Smoke] Annotation-like files: {len(txt_like):,}')
        print(f'[Smoke] Image files: {len(img_like):,}')

        print('[Smoke] Annotation-like sample list:')
        for p in sorted(txt_like)[:40]:
            print(' -', p)
    except Exception as diag_e:
        print('[Smoke] Diagnostic step failed:', repr(diag_e))

    raise

## 4) Full Training
Start full training after smoke test succeeds.

In [ ]:
# Full training run
cfg = TrainConfig(epochs=60, batch_size=16, lr=1e-3)
run_training(cfg)

In [ ]:
# List output files
import os

out_dir = '/kaggle/working/fullpage_saved_models'
print('Output dir exists:', os.path.isdir(out_dir))
if os.path.isdir(out_dir):
    for name in sorted(os.listdir(out_dir)):
        p = os.path.join(out_dir, name)
        size_mb = os.path.getsize(p) / (1024 * 1024)
        print(f'{name:40s} {size_mb:8.2f} MB')

# Zip outputs for easy download
if os.path.isdir(out_dir):
    !cd /kaggle/working && zip -r fullpage_saved_models.zip fullpage_saved_models

## 5) Files to Download
Download from Kaggle Output:
- `fullpage_saved_models.zip` (recommended)
or individually:
- `crnn_iam_inference.keras`
- `char_map.json`
- `crnn_iam_best.weights.h5`
- `training.csv`

Then share the first two files for app integration.